# RAGAS RAG evaluation (English UDHR)

**What you will learn**

- Run a small **retrieve → generate** pipeline on the English [UDHR](https://www.un.org/en/about-us/universal-declaration-of-human-rights) corpus
- Score it with **[RAGAS](https://docs.ragas.io/)** — four metrics split into **retrieval** vs **generation**
- Compare RAGAS scores with a simple **rank-1 article** check (outside RAGAS)

**Corpus:** `data/udhr_en.txt` (preamble + 30 articles, ~61 chunks)

**Stack:** [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) embeddings + [LangChain Chroma](https://python.langchain.com/docs/integrations/vectorstores/chroma/) + OpenAI (`gpt-4o-mini`) for answers and RAGAS judging

**Prerequisites** (from repo root):

```bash
uv sync
# Option A: repo-root .env (gitignored) with OPENAI_API_KEY=...
# Option B: export OPENAI_API_KEY="your-key"
uv run jupyter lab experiments/ragas-rag-evaluation/notebook.ipynb
```

The first code cell loads `OPENAI_API_KEY` from repo-root `.env` via [python-dotenv](https://github.com/theskumar/python-dotenv).

Run all cells **top to bottom**.

**First run:** MiniLM downloads from Hugging Face (~**80 MB**; cached under `~/.cache/huggingface/hub/`). Ingest embeds ~61 chunks on CPU. RAGAS evaluation calls OpenAI several times per eval row (cost + latency).


## Concepts (quick links)

| Idea | Link |
|------|------|
| RAG (retrieve, then generate) | [LangChain RAG](https://python.langchain.com/docs/concepts/rag/) |
| RAGAS metrics | [Available metrics](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/) |
| MiniLM embeddings | [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) |
| Vector database (Chroma) | [Chroma docs](https://docs.trychroma.com/) |

**Pipeline:** load text → chunk by article → embed → store in Chroma → for each eval question: retrieve → generate answer → score with RAGAS.


## 1. Locate data, index, and eval-set paths

Resolve `EXPERIMENT_DIR` (works when Jupyter cwd is repo root or this folder), then set paths for the UDHR file, on-disk Chroma index (`vector_db/`, gitignored), and the eval JSONL.


In [3]:
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

# Jupyter cwd may be repo root or this experiment folder — detect via notebook.ipynb.
EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "notebook.ipynb").exists():
    alt = Path("experiments/ragas-rag-evaluation").resolve()
    if (alt / "notebook.ipynb").exists():
        EXPERIMENT_DIR = alt


DATA_DIR = EXPERIMENT_DIR / "data"
VECTOR_DB_PATH = EXPERIMENT_DIR / "vector_db"
EVAL_SET_PATH = DATA_DIR / "eval_set.jsonl"
UDHR_PATH = DATA_DIR / "udhr_en.txt"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Vector DB path: {VECTOR_DB_PATH}")
print(f"Eval set: {EVAL_SET_PATH}")


Experiment dir: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/ragas-rag-evaluation
Vector DB path: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/ragas-rag-evaluation/vector_db
Eval set: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/ragas-rag-evaluation/data/eval_set.jsonl


## 2. Define helpers — MiniLM embeddings, article chunking, retrieval, generation

This cell **defines** functions only (no model load yet). Ingest runs in the next section.

- **MiniLM:** same embedding for index and queries (no prefix required)
- **`article` metadata:** used for rank-1 asserts and result tables
- **`reference` in eval set:** gold answer text for RAGAS retrieval metrics


In [5]:
import os
import re
import shutil

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI

COLLECTION_NAME = "ragas_udhr_en"
# Small English embedder — fast first run (~80 MB HF download).
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
# OpenAI model for (1) grounded answers and (2) RAGAS LLM-as-judge metrics.
GENERATION_MODEL = "gpt-4o-mini"
# Number of chunks passed to the generator and to RAGAS context_* metrics.
RETRIEVAL_K = 5

# Split on English "Article N" headings.
ARTICLE_MARKER = re.compile(r"(?=\bArticle\s+(\d+)\b)", re.IGNORECASE)
ARTICLE_ID_FROM_START = re.compile(r"^\s*Article\s+(\d+)\b", re.IGNORECASE)


def _article_id(chunk: str) -> str:
    match = ARTICLE_ID_FROM_START.search(chunk.strip())
    return match.group(1) if match else "preamble"


def load_udhr_text() -> str:
    if not UDHR_PATH.exists():
        raise FileNotFoundError(f"Missing {UDHR_PATH}")
    return UDHR_PATH.read_text(encoding="utf-8")


def split_by_articles(text: str) -> list[Document]:
    parts = ARTICLE_MARKER.split(text)
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) > 1:
        docs = []
        for part in parts:
            docs.append(
                Document(
                    page_content=part,
                    metadata={"article": _article_id(part), "source": "udhr"},
                )
            )
        return docs
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    return splitter.create_documents(
        [text],
        metadatas=[{"article": "unknown", "source": "udhr"}],
    )


def build_vector_db(documents: list[Document], *, reset: bool = True) -> Chroma:
    """Create or replace the on-disk vector index (Chroma backend)."""
    if reset and VECTOR_DB_PATH.exists():
        shutil.rmtree(VECTOR_DB_PATH)
    VECTOR_DB_PATH.mkdir(parents=True, exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    return Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
    )


def retrieve_for_eval(
    vector_db: Chroma,
    query: str,
    *,
    k: int = RETRIEVAL_K,
) -> tuple[list[str], list[str], pd.DataFrame]:
    """Return context texts, article ids per hit, and a display table."""
    hits = vector_db.similarity_search_with_score(query, k=k)
    contexts: list[str] = []
    articles: list[str] = []
    rows = []
    for rank, (doc, score) in enumerate(hits, start=1):
        contexts.append(doc.page_content)
        article = str(doc.metadata.get("article", ""))
        articles.append(article)
        preview = doc.page_content[:120] + "..." if len(doc.page_content) > 120 else doc.page_content
        rows.append(
            {
                "rank": rank,
                "article": article,
                "score": round(float(score), 4),
                "text": preview,
            }
        )
    return contexts, articles, pd.DataFrame(rows)


def assert_rank1_article(articles: list[str], expected_article: str, *, query: str) -> None:
    """Deterministic retrieval check — not a RAGAS metric."""
    assert articles, f"No results for query: {query!r}"
    assert articles[0] == expected_article, (
        f"Expected article {expected_article} at rank 1, got {articles[0]!r} for: {query!r}"
    )


def generate_answer(client: OpenAI, question: str, contexts: list[str]) -> str:
    """Grounded answer from retrieved chunks only (used as RAGAS `response`)."""
    context_block = "\n\n---\n\n".join(contexts)
    completion = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the provided context. "
                    "If the context does not contain the answer, say you do not know."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context_block}\n\nQuestion: {question}",
            },
        ],
        temperature=0,
    )
    return completion.choices[0].message.content or ""


## 3. Ingest — chunk, embed, and persist the English index

Loads `udhr_en.txt`, splits into ~61 article-level chunks, embeds with MiniLM, and writes `vector_db/`. Expect chunk count printout and encoding progress bars on first run.


In [6]:
text = load_udhr_text()
documents = split_by_articles(text)
print(f"Chunks to index: {len(documents)}")

vector_db = build_vector_db(documents, reset=True)
print(f"Indexed {len(documents)} chunks into {COLLECTION_NAME!r}")


Chunks to index: 61


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 61 chunks into 'ragas_udhr_en'


## 4. What RAGAS measures — retrieval vs generation

RAGAS scores different stages of a RAG pipeline. All scores are **0 to 1** (higher is better).

### Retrieval-based metrics (did we fetch the right chunks?)

These judge the **retriever** using `user_input`, `retrieved_contexts`, and gold `reference`.

| Metric | What it asks | Plain language |
|--------|----------------|----------------|
| [Context Precision](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_precision/) | Are relevant chunks ranked above irrelevant ones? | Did we put the good passages first? |
| [Context Recall](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/) | Do contexts cover the reference answer? | Did we retrieve enough of the right information? |

> **Retrieval evals** matter even when generation fails. Low precision/recall → fix embeddings, chunking, or `k` before tuning prompts.

**Outside RAGAS:** we also `assert` rank-1 `article` metadata matches `expected_article` in `eval_set.jsonl` — a fast sanity check like the [multilingual retrieval notebook](../multilingual-rag-retrieval/notebook.ipynb).

### Generation-based metrics (did the LLM answer well?)

These judge the **generator** using `user_input`, `response`, and `retrieved_contexts`.

| Metric | What it asks | Plain language |
|--------|----------------|----------------|
| [Faithfulness](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/) | Is the answer supported by the contexts? | Did the model hallucinate? |
| [Answer Relevancy](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_relevance/) | Does the answer address the question? | Did it answer what was asked? |

> **Generation evals** assume retrieval already ran. Low faithfulness → model ignored context; low relevancy → vague or off-topic answer.

```
Question → [Retriever] → contexts → [LLM] → answer
              ↑                        ↑
     Context Precision/Recall      Faithfulness
        (retrieval evals)          Answer Relevancy
                                   (generation evals)
```


## 5. Load the eval set — fields and who uses them

`data/eval_set.jsonl` has one JSON object per line:

| Field | Used by |
|-------|---------|
| `user_input` | Retrieval query + all four RAGAS metrics |
| `reference` | **Context Precision** and **Context Recall** (gold answer) |
| `expected_article` | Gold article for comparison (e.g. `"3"`) |
| `expect_rank1` | If `true`, we **assert** rank-1 matches `expected_article` (direct paraphrases only) |

Rows with `expect_rank1: false` use indirect wording (articles 7, 19). Rank-1 may miss with MiniLM — compare `rank1_ok` and RAGAS retrieval scores.


In [7]:
eval_df = pd.read_json(EVAL_SET_PATH, lines=True)
print(f"Eval rows: {len(eval_df)}")
display(eval_df)


Eval rows: 7


,user_input,reference,expected_article,expect_rank1
0,All people are born free and equal in dignity ...,All human beings are born free and equal in di...,1,True
1,Rights in this declaration apply to everyone w...,Everyone is entitled to all the rights and fre...,2,True
2,"Everyone has the right to life, liberty and se...","Everyone has the right to life, liberty and se...",3,True
3,Can someone be held as a slave under this decl...,No one shall be held in slavery or servitude; ...,4,True
4,What does the declaration say about torture?,No one shall be subjected to torture or to cru...,5,True
5,Are men and women treated the same under the law?,All are equal before the law and are entitled ...,7,False
6,Do people have the right to share their opinio...,Everyone has the right to freedom of opinion a...,19,False


## 6. Run the pipeline — retrieve, check rank-1, generate, build RAGAS samples

For each eval row we:

1. **Retrieve** top-`k` chunks from Chroma
2. Record whether rank-1 `article` matches `expected_article` (`rank1_ok`)
3. **Assert** rank-1 only when `expect_rank1` is `true` (direct paraphrases)
4. **Generate** an answer with OpenAI from those contexts
5. Build a `SingleTurnSample` for RAGAS (`user_input`, `retrieved_contexts`, `response`, `reference`)


In [8]:
import os

from ragas import SingleTurnSample

if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY before running generation or RAGAS eval.")

client = OpenAI()
pipeline_rows: list[dict] = []
ragas_samples: list[SingleTurnSample] = []

for _, row in eval_df.iterrows():
    question = row["user_input"]
    reference = row["reference"]
    expected_article = str(row["expected_article"])
    expect_rank1 = bool(row["expect_rank1"])

    contexts, articles, hit_df = retrieve_for_eval(vector_db, question, k=RETRIEVAL_K)
    rank1_ok = articles[0] == expected_article

    # Deterministic retrieval check (not RAGAS) — only for direct paraphrases.
    if expect_rank1:
        assert_rank1_article(articles, expected_article, query=question)

    answer = generate_answer(client, question, contexts)

    pipeline_rows.append(
        {
            "user_input": question,
            "expected_article": expected_article,
            "rank1_article": articles[0],
            "rank1_ok": rank1_ok,
            "expect_rank1": expect_rank1,
            "response_preview": answer[:160] + ("..." if len(answer) > 160 else ""),
        }
    )

    # Fields RAGAS expects on each sample.
    ragas_samples.append(
        SingleTurnSample(
            user_input=question,
            retrieved_contexts=contexts,
            response=answer,
            reference=reference,
        )
    )

pipeline_df = pd.DataFrame(pipeline_rows)
display(pipeline_df)
print(f"Built {len(ragas_samples)} RAGAS samples")
print("Rank-1 pass rate:", pipeline_df["rank1_ok"].mean())


,user_input,expected_article,rank1_article,rank1_ok,expect_rank1,response_preview
0,All people are born free and equal in dignity ...,1,1,True,True,True. All human beings are born free and equal...
1,Rights in this declaration apply to everyone w...,2,2,True,True,"Yes, rights in this declaration apply to every..."
2,"Everyone has the right to life, liberty and se...",3,3,True,True,This statement is found in Article 3 of the pr...
3,Can someone be held as a slave under this decl...,4,4,True,True,"No, someone cannot be held as a slave under th..."
4,What does the declaration say about torture?,5,5,True,True,The declaration states that no one shall be su...
5,Are men and women treated the same under the law?,7,16,False,False,"Yes, men and women are treated the same under ..."
6,Do people have the right to share their opinio...,19,19,True,False,"Yes, people have the right to share their opin..."


Built 7 RAGAS samples
Rank-1 pass rate: 0.8571428571428571


## 7. RAGAS evaluation — four metrics with OpenAI judge

`evaluate()` runs all metrics on the sample list. We group them explicitly:

- **Retrieval:** Context Precision, Context Recall
- **Generation:** Faithfulness, Answer Relevancy

This may take a few minutes (multiple LLM calls per row).


In [9]:
from ragas import EvaluationDataset, evaluate
from ragas.llms import llm_factory
from ragas.metrics import (
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
)

# EvaluationDataset: one row per eval question with fields each metric needs.
eval_dataset = EvaluationDataset.from_list([s.to_dict() for s in ragas_samples])

# RAGAS uses this LLM as judge (reads OPENAI_API_KEY) — separate from generate_answer calls above.
evaluator_llm = llm_factory(GENERATION_MODEL)

retrieval_metrics = [
    ContextPrecision(),  # retrieval: relevant chunks ranked high?
    ContextRecall(),  # retrieval: contexts cover reference?
]
generation_metrics = [
    Faithfulness(),  # generation: answer grounded in contexts?
    AnswerRelevancy(),  # generation: answer addresses question?
]
metrics = retrieval_metrics + generation_metrics

result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    show_progress=True,
)

scores_df = result.to_pandas()
# Friendly column names for beginners.
rename_map = {
    "context_precision": "context_precision (retrieval)",
    "context_recall": "context_recall (retrieval)",
    "faithfulness": "faithfulness (generation)",
    "answer_relevancy": "answer_relevancy (generation)",
}
scores_df = scores_df.rename(columns={k: v for k, v in rename_map.items() if k in scores_df.columns})
display(scores_df)

print("\nMean scores:")
for col in scores_df.columns:
    if col in rename_map.values() or col.endswith(")"):
        print(f"  {col}: {scores_df[col].mean():.3f}")


Evaluating:   0%|          | 0/28 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,context_precision (retrieval),context_recall (retrieval),faithfulness (generation),answer_relevancy (generation)
0,All people are born free and equal in dignity ...,[Article 1\nAll human beings are born free and...,True. All human beings are born free and equal...,All human beings are born free and equal in di...,0.805556,1.0,1.0,0.832101
1,Rights in this declaration apply to everyone w...,[Article 2\nEveryone is entitled to all the ri...,"Yes, rights in this declaration apply to every...",Everyone is entitled to all the rights and fre...,0.804167,1.0,1.0,0.914548
2,"Everyone has the right to life, liberty and se...","[Article 3\nEveryone has the right to life, li...",This statement is found in Article 3 of the pr...,"Everyone has the right to life, liberty and se...",0.700000,1.0,1.0,0.761055
3,Can someone be held as a slave under this decl...,[Article 4\nNo one shall be held in slavery or...,"No, someone cannot be held as a slave under th...",No one shall be held in slavery or servitude; ...,1.000000,1.0,1.0,1.000000
4,What does the declaration say about torture?,[Article 5\nNo one shall be subjected to tortu...,The declaration states that no one shall be su...,No one shall be subjected to torture or to cru...,1.000000,1.0,1.0,0.977175
5,Are men and women treated the same under the law?,"[Article 16\n 1. Men and women of full age, wi...","Yes, men and women are treated the same under ...",All are equal before the law and are entitled ...,0.950000,1.0,1.0,0.982490
6,Do people have the right to share their opinio...,[Article 19\nEveryone has the right to freedom...,"Yes, people have the right to share their opin...",Everyone has the right to freedom of opinion a...,1.000000,1.0,1.0,1.000000



Mean scores:
  context_precision (retrieval): 0.894
  context_recall (retrieval): 1.000
  faithfulness (generation): 1.000
  answer_relevancy (generation): 0.924


## 8. Interpret results — RAGAS scores vs rank-1 asserts

Use the tables above together:

- **Rank-1 assert passed** but low **context precision** → right article, but noisy neighbors in top-`k`
- **Rank-1 assert failed** → retrieval missed; expect lower retrieval metrics; fix embedder/chunking/`k` before prompt tuning
- Low **faithfulness** with good retrieval → model ignored context; tighten generation prompt
- Low **answer relevancy** with high faithfulness → grounded but off-topic or too short

Rows with `expect_rank1: true` should pass asserts with MiniLM on this corpus. Rows with `expect_rank1: false` may show `rank1_ok: false` — compare with RAGAS retrieval scores (not a RAGAS bug).


In [10]:
# One table: pipeline metadata + per-row RAGAS scores.
combined = pd.concat(
    [pipeline_df.reset_index(drop=True), scores_df.reset_index(drop=True)],
    axis=1,
)
display(combined)


,user_input,expected_article,rank1_article,rank1_ok,expect_rank1,response_preview,user_input,retrieved_contexts,response,reference,context_precision (retrieval),context_recall (retrieval),faithfulness (generation),answer_relevancy (generation)
0,All people are born free and equal in dignity ...,1,1,True,True,True. All human beings are born free and equal...,All people are born free and equal in dignity ...,[Article 1\nAll human beings are born free and...,True. All human beings are born free and equal...,All human beings are born free and equal in di...,0.805556,1.0,1.0,0.832101
1,Rights in this declaration apply to everyone w...,2,2,True,True,"Yes, rights in this declaration apply to every...",Rights in this declaration apply to everyone w...,[Article 2\nEveryone is entitled to all the ri...,"Yes, rights in this declaration apply to every...",Everyone is entitled to all the rights and fre...,0.804167,1.0,1.0,0.914548
2,"Everyone has the right to life, liberty and se...",3,3,True,True,This statement is found in Article 3 of the pr...,"Everyone has the right to life, liberty and se...","[Article 3\nEveryone has the right to life, li...",This statement is found in Article 3 of the pr...,"Everyone has the right to life, liberty and se...",0.700000,1.0,1.0,0.761055
3,Can someone be held as a slave under this decl...,4,4,True,True,"No, someone cannot be held as a slave under th...",Can someone be held as a slave under this decl...,[Article 4\nNo one shall be held in slavery or...,"No, someone cannot be held as a slave under th...",No one shall be held in slavery or servitude; ...,1.000000,1.0,1.0,1.000000
4,What does the declaration say about torture?,5,5,True,True,The declaration states that no one shall be su...,What does the declaration say about torture?,[Article 5\nNo one shall be subjected to tortu...,The declaration states that no one shall be su...,No one shall be subjected to torture or to cru...,1.000000,1.0,1.0,0.977175
5,Are men and women treated the same under the law?,7,16,False,False,"Yes, men and women are treated the same under ...",Are men and women treated the same under the law?,"[Article 16\n 1. Men and women of full age, wi...","Yes, men and women are treated the same under ...",All are equal before the law and are entitled ...,0.950000,1.0,1.0,0.982490
6,Do people have the right to share their opinio...,19,19,True,False,"Yes, people have the right to share their opin...",Do people have the right to share their opinio...,[Article 19\nEveryone has the right to freedom...,"Yes, people have the right to share their opin...",Everyone has the right to freedom of opinion a...,1.000000,1.0,1.0,1.000000


## 9. Wrap-up

**You ran:** English UDHR index (MiniLM + Chroma) → retrieve → generate → **RAGAS** (2 retrieval + 2 generation metrics) plus rank-1 article asserts.

**When to fix what**

| Symptom | Likely layer |
|---------|----------------|
| Low context precision / recall | Retriever (embeddings, chunks, `k`) |
| Rank-1 assert failures | Same as above |
| Low faithfulness | Generator ignoring context |
| Low answer relevancy | Generator off-topic or incomplete |

**Tradeoffs:** MiniLM is small and fast but weaker on hard paraphrases than larger English embedders. RAGAS LLM metrics add cost and latency — use a small eval set while learning.

**Next:** [multilingual-rag-retrieval](../multilingual-rag-retrieval/notebook.ipynb) for bilingual retrieval and metadata filters (no RAGAS).
